# Семинар: Введение в PyTorch и автоматическое дифференцирование

![img](https://pytorch.org/tutorials/_static/pytorch-logo-dark.svg)

В этом ноутбуке мы познакомимся с основами PyTorch — одного из самых популярных фреймворков для глубокого обучения.

**Что мы изучим:**
1. Тензоры PyTorch и их связь с NumPy
2. Автоматическое вычисление градиентов (autograd)
3. Линейная регрессия на PyTorch
4. Высокоуровневый API: `torch.nn`
5. Обучение модели классификации

**Главное отличие PyTorch от TensorFlow 1.x:**  
В PyTorch нет разделения на "символьный граф" и "реальные данные". Все тензоры сразу содержат числовые значения. Код выглядит практически как обычный NumPy, но PyTorch автоматически считает градиенты и умеет работать с GPU.

In [ ]:
# Установка зависимостей (раскомментировать при необходимости)
# !pip install torch torchvision matplotlib numpy pandas scikit-learn scikit-image

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
%matplotlib inline

print(f"PyTorch version: {torch.__version__}")

---
## 1. Тензоры: NumPy vs PyTorch

Тензоры в PyTorch — это аналог `ndarray` в NumPy. Большинство операций имеют похожий синтаксис, но есть отличия:

| NumPy | PyTorch | Описание |
|-------|---------|----------|
| `np.array([1,2,3])` | `torch.tensor([1,2,3])` | Создание тензора |
| `x.reshape(2,3)` | `x.view(2,3)` или `x.reshape(2,3)` | Изменение формы |
| `x.sum(axis=0)` | `x.sum(dim=0)` | Суммирование по оси |
| `np.dot(x, y)` | `torch.matmul(x, y)` или `x @ y` | Матричное умножение |
| `x.astype('float32')` | `x.float()` или `x.to(torch.float32)` | Приведение типа |
| `x` (ndarray) | `x.numpy()` | Конвертация в NumPy |

In [ ]:
# --- NumPy ---
x_np = np.arange(16).reshape(4, 4)

print("=== NumPy ===")
print("X:\n", x_np)
print("shape:", x_np.shape)
print("X + 5:\n", x_np + 5)
print("X @ X^T:\n", x_np @ x_np.T)
print("mean по столбцам:", x_np.mean(axis=-1))

In [ ]:
# --- PyTorch ---
x_pt = torch.arange(16, dtype=torch.float32).view(4, 4)

print("=== PyTorch ===")
print("X:\n", x_pt)
print("shape:", x_pt.shape)
print("X + 5:\n", x_pt + 5)
print("X @ X^T:\n", x_pt @ x_pt.T)  # то же, что torch.matmul(x_pt, x_pt.T)
print("mean по столбцам:", x_pt.mean(dim=-1))

### Задание 1: Тригонометрический узор

Постройте параметрическую кривую:

$$ x(t) = t - 1.5 \cdot \cos(15t) $$
$$ y(t) = t - 1.5 \cdot \sin(16t) $$

Используйте `torch.linspace`, `torch.cos`, `torch.sin`.

In [ ]:
t = torch.linspace(-10, 10, steps=10000)

# Вычислите x(t) и y(t)
x = # YOUR CODE
y = # YOUR CODE

plt.figure(figsize=(8, 8))
plt.plot(x.numpy(), y.numpy(), linewidth=0.5)
plt.title('Тригонометрический узор')
plt.show()

**Бонус:** попробуйте поменять коэффициенты (15 и 16) и посмотрите, как меняется узор.

---
## 2. Автоматическое дифференцирование (autograd)

Любой фреймворк глубокого обучения должен уметь считать градиенты автоматически. В PyTorch это делает модуль `autograd`.

**Алгоритм:**
1. Создаём тензор с флагом `requires_grad=True`
2. Производим вычисления (строим вычислительный граф)
3. Вызываем `.backward()` у скалярного результата
4. Градиенты доступны в `.grad`

### Пример: простые градиенты

In [ ]:
# Пример: f(x) = x^2, f'(x) = 2x
x = torch.tensor(3.0, requires_grad=True)
f = x ** 2
f.backward()

print(f"x = {x.item()}, f(x) = x^2 = {f.item()}, df/dx = 2x = {x.grad.item()}")

### Задание 2: Вычислите градиенты вручную и проверьте через autograd

Дана функция:
$$ f(x, y) = x^2 y + y^3 + 2xy $$

1. Вычислите $\frac{\partial f}{\partial x}$ и $\frac{\partial f}{\partial y}$ аналитически (на бумаге)
2. Проверьте результат с помощью PyTorch в точке $(x, y) = (2, 3)$

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

f = # YOUR CODE: x^2 * y + y^3 + 2*x*y

f.backward()

print(f"df/dx (PyTorch) = {x.grad.item()}")
print(f"df/dy (PyTorch) = {y.grad.item()}")

# Подставьте свои аналитические ответы:
# df/dx = YOUR ANSWER (подставьте x=2, y=3)
# df/dy = YOUR ANSWER (подставьте x=2, y=3)

---
## 3. Линейная регрессия на PyTorch

Обучим линейную регрессию $\hat{y} = wx + b$ методом градиентного спуска.

Используем данные Boston Housing (предсказание цен на жильё).

In [ ]:
# Загрузка данных
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

plt.scatter(data[:, -1], target)
plt.xlabel('LSTAT (% населения с низким статусом)')
plt.ylabel('Цена жилья ($1000)')
plt.title('Boston Housing')
plt.show()

In [ ]:
# Инициализация параметров
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)

# Данные (нормализуем признак для стабильности)
x = torch.tensor(data[:, -1] / 10, dtype=torch.float32)
y = torch.tensor(target, dtype=torch.float32)

In [ ]:
# Один шаг: предсказание, функция потерь, backward
y_pred = w * x + b
loss = torch.mean((y_pred - y) ** 2)  # MSE
loss.backward()

print("dL/dw =", w.grad)
print("dL/db =", b.grad)

### Задание 3: Обучите линейную регрессию

Реализуйте цикл обучения:
1. Предсказание: $\hat{y} = wx + b$
2. Функция потерь: MSE $= \frac{1}{N}\sum (\hat{y}_i - y_i)^2$
3. `loss.backward()` — вычисление градиентов
4. Обновление весов: $w \leftarrow w - \alpha \cdot \frac{\partial L}{\partial w}$
5. Обнуление градиентов: `.grad.zero_()`

**Важно:** обновление весов нужно делать в блоке `with torch.no_grad():`, чтобы оно не попало в вычислительный граф.

In [ ]:
from IPython.display import clear_output

# Заново инициализируем параметры
w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
learning_rate = 0.05

for i in range(100):
    # 1. Предсказание
    y_pred = # YOUR CODE

    # 2. Функция потерь (MSE)
    loss = # YOUR CODE

    # 3. Вычисление градиентов
    # YOUR CODE

    # 4. Обновление весов (без отслеживания градиентов!)
    with torch.no_grad():
        # YOUR CODE: обновить w и b
        # YOUR CODE: обнулить градиенты
        pass

    # Визуализация каждые 5 шагов
    if (i + 1) % 5 == 0:
        clear_output(True)
        plt.scatter(x.data.numpy(), y.data.numpy(), alpha=0.5)
        plt.scatter(x.data.numpy(), y_pred.data.numpy(), color='orange', s=5)
        plt.title(f'Шаг {i+1}, MSE = {loss.item():.3f}')
        plt.show()

        if loss.item() < 0.5:
            print("Готово!")
            break

---
## 4. Высокоуровневый API: torch.nn

Для построения нейросетей вручную управлять каждым тензором неудобно. PyTorch предоставляет модуль `torch.nn` с готовыми слоями и моделями.

Основные концепции:
- **`nn.Module`** — базовый класс для всех моделей и слоёв
- **`nn.Sequential`** — контейнер для последовательного применения слоёв
- **`nn.Linear(in, out)`** — полносвязный слой
- **`torch.optim`** — оптимизаторы (SGD, Adam, RMSprop, ...)

### Задача: бинарная классификация букв A vs B

Будем обучать логистическую регрессию:
$$P(y=1 | X) = \sigma(W \cdot X + b) = \frac{1}{1 + e^{-(W \cdot X + b)}}$$

In [ ]:
# Загрузка данных notMNIST (буквы A и B)
# Если файл notmnist.py отсутствует, скачайте:
# !wget https://raw.githubusercontent.com/yandexdataschool/Practical_DL/fall19/week02_autodiff/notmnist.py -O notmnist.py

from notmnist import load_notmnist
X_train, y_train, X_test, y_test = load_notmnist(letters='AB')
X_train, X_test = X_train.reshape([-1, 784]), X_test.reshape([-1, 784])

print(f"Размер обучающей выборки: {len(X_train)}, тестовой: {len(X_test)}")

# Визуализация примеров
fig, axes = plt.subplots(1, 4, figsize=(10, 3))
for i, ax in enumerate(axes):
    ax.imshow(X_train[i].reshape(28, 28), cmap='gray')
    ax.set_title(f'Класс: {y_train[i]} ({"A" if y_train[i]==0 else "B"})')
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
from torch import nn
import torch.nn.functional as F

# Создаём модель: логистическая регрессия
model = nn.Sequential(
    nn.Linear(784, 1),  # полносвязный слой: 784 входов -> 1 выход
    nn.Sigmoid()        # сигмоида для получения вероятности
)

print("Параметры модели:", [(name, p.shape) for name, p in model.named_parameters()])

In [ ]:
# Проверяем, что модель работает
x_sample = torch.tensor(X_train[:3], dtype=torch.float32)
y_sample = torch.tensor(y_train[:3], dtype=torch.float32)

y_predicted = model(x_sample)[:, 0]  # [:, 0] чтобы убрать лишнюю размерность
print("Предсказанные вероятности (до обучения):", y_predicted.data)

### Задание 4: Реализуйте бинарную кросс-энтропию

Функция потерь для бинарной классификации:

$$ L = -\frac{1}{N} \sum_{i=1}^{N} \left[ y_i \cdot \log P(y_i|X_i) + (1 - y_i) \cdot \log(1 - P(y_i|X_i)) \right] $$

Реализуйте вручную **без** использования `torch.nn.functional.binary_cross_entropy`.

In [ ]:
# Вычислите кросс-энтропию для каждого элемента (вектор длины N)
crossentropy = # YOUR CODE: -( y * log(p) + (1-y) * log(1-p) )

# Среднее значение — скалярная функция потерь
loss = # YOUR CODE: среднее от crossentropy

# Проверки
assert tuple(crossentropy.size()) == (3,), "crossentropy должен быть вектором длины N"
assert tuple(loss.size()) == tuple(), "loss должен быть скаляром (не забыли mean?)"
assert loss.item() > 0, "Кросс-энтропия всегда >= 0"
assert loss.item() <= np.log(3), "Loss слишком большой для необученной модели"
print(f"Loss = {loss.item():.4f} — всё верно!")

### Оптимизатор

Вместо ручного обновления весов удобно использовать `torch.optim`:

In [ ]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

# Стандартный цикл:
# loss.backward()    — вычислить градиенты
# opt.step()         — обновить веса
# opt.zero_grad()    — обнулить градиенты

In [ ]:
# Очистка переменных перед основным обучением
del x_sample, y_sample, y_predicted, loss, crossentropy

### Задание 5: Обучите модель

Заполните пропуски в цикле обучения:

In [ ]:
# Создаём модель и оптимизатор заново
model = nn.Sequential(
    nn.Linear(784, 1),
    nn.Sigmoid()
)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

history = []

for i in range(100):
    # Случайный мини-батч из 256 примеров
    ix = np.random.randint(0, len(X_train), 256)
    x_batch = torch.tensor(X_train[ix], dtype=torch.float32)
    y_batch = torch.tensor(y_train[ix], dtype=torch.float32)

    # Предсказание (не забудьте [:, 0])
    y_predicted = # YOUR CODE

    assert y_predicted.dim() == 1, "Не забыли [:, 0]?"

    # Вычислите loss (бинарная кросс-энтропия)
    loss = # YOUR CODE

    # Вычислите градиенты
    # YOUR CODE

    # Шаг оптимизатора
    # YOUR CODE

    # Обнулите градиенты
    # YOUR CODE

    history.append(loss.item())

    if i % 10 == 0:
        print(f"Шаг {i:3d} | mean loss = {np.mean(history[-10:]):.3f}")

plt.plot(history)
plt.xlabel('Шаг')
plt.ylabel('Loss')
plt.title('Кривая обучения')
plt.show()

**Советы по отладке:**
- Проверяйте, что модель выдаёт вероятности (числа от 0 до 1)
- Не забудьте знак **минус** в формуле кросс-энтропии
- Обнуляйте градиенты после каждого шага
- Если loss стал `nan`, попробуйте уменьшить learning rate

### Задание 6: Оценка качества на тестовой выборке

Получите предсказания модели для тестовых данных и вычислите accuracy.

In [ ]:
# Получите предсказания классов (0 или 1) для тестовой выборки
# Подсказка: model(X_test) даст вероятности, порог 0.5
predicted_y_test = # YOUR CODE (должен быть np.ndarray)

assert isinstance(predicted_y_test, np.ndarray), f"Нужен np.ndarray, а не {type(predicted_y_test)}"
assert predicted_y_test.shape == y_test.shape, "Размер предсказаний должен совпадать с y_test"

accuracy = np.mean(predicted_y_test == y_test)
print(f"Accuracy на тесте: {accuracy:.4f}")
assert accuracy > 0.95, "Accuracy должен быть > 0.95. Попробуйте обучить дольше."

---
## 5. Бонусное задание: нелинейная модель

Попробуйте создать нейросеть с одним или несколькими скрытыми слоями и улучшить accuracy.

Пример архитектуры:
```python
model = nn.Sequential(
    nn.Linear(784, 64),
    nn.ReLU(),
    nn.Linear(64, 1),
    nn.Sigmoid()
)
```

In [ ]:
# YOUR CODE: создайте нейросеть, обучите и оцените

---
## 6. Работа с изображениями в Python

Прежде чем переходить к свёрточным сетям, важно понять, как загружать и обрабатывать изображения.

### Основные библиотеки

| Библиотека | Формат данных | Порядок каналов | Тип значений |
|-----------|--------------|----------------|-------------|
| PIL/Pillow | PIL.Image (объект) | RGB | — |
| NumPy | (H, W, C) ndarray | RGB | uint8 [0-255] / float |
| OpenCV (cv2) | (H, W, C) ndarray | **BGR** | uint8 [0-255] |
| matplotlib | (H, W, C) ndarray | RGB | любой |
| **PyTorch** | **(C, H, W)** тензор | RGB | float32 [0-1] |

**Ключевой момент:** PyTorch ожидает каналы первыми — **(C, H, W)**, а не (H, W, C) как все остальные!

In [ ]:
from PIL import Image
from torchvision import transforms
import urllib.request
from io import BytesIO

# Загрузим тестовое изображение
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/4/4d/Cat_November_2010-1a.jpg/1200px-Cat_November_2010-1a.jpg"
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
img_pil = Image.open(BytesIO(urllib.request.urlopen(req).read())).convert("RGB")

print(f"PIL Image: size={img_pil.size}, mode={img_pil.mode}")
plt.imshow(img_pil)
plt.title("PIL Image")
plt.axis('off')
plt.show()

In [ ]:
# PIL -> NumPy
img_np = np.array(img_pil)
print(f"NumPy: shape={img_np.shape}, dtype={img_np.dtype}, range=[{img_np.min()}, {img_np.max()}]")
# (H, W, C), uint8, [0, 255]

In [ ]:
# PIL -> PyTorch через transforms.ToTensor()
to_tensor = transforms.ToTensor()
img_tensor = to_tensor(img_pil)
print(f"Tensor: shape={img_tensor.shape}, dtype={img_tensor.dtype}, range=[{img_tensor.min():.2f}, {img_tensor.max():.2f}]")
# (C, H, W), float32, [0.0, 1.0]

`transforms.ToTensor()` делает три вещи:
1. PIL/NumPy → `torch.Tensor`
2. (H, W, C) → **(C, H, W)**
3. uint8 [0, 255] → float32 **[0.0, 1.0]**

In [ ]:
# Полный пайплайн для нейросети
preprocess = transforms.Compose([
    transforms.Resize(256),                          # изменить размер
    transforms.CenterCrop(224),                      # обрезать 224x224 по центру
    transforms.ToTensor(),                           # PIL -> (C,H,W) float32
    transforms.Normalize(                            # нормализация ImageNet
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

img_processed = preprocess(img_pil)
print(f"После пайплайна: shape={img_processed.shape}, range=[{img_processed.min():.2f}, {img_processed.max():.2f}]")

# Для визуализации нужно "развернуть" нормализацию
def denormalize(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    t = tensor.clone()
    for c in range(3):
        t[c] = t[c] * std[c] + mean[c]
    return t.permute(1, 2, 0).clamp(0, 1).numpy()

plt.imshow(denormalize(img_processed))
plt.title('После Resize + CenterCrop + Normalize')
plt.axis('off')
plt.show()

### Задание 7: Загрузка и обработка изображения

1. Загрузите любое изображение через PIL (можно локальное или по URL)
2. Переведите в NumPy, выведите shape и dtype
3. Переведите в тензор через `transforms.ToTensor()`, проверьте shape
4. Примените `transforms.RandomHorizontalFlip(p=1.0)` и `transforms.RandomRotation(30)` к PIL-изображению и покажите результат

In [ ]:
# YOUR CODE

# 1. Загрузите изображение

# 2. NumPy: shape, dtype

# 3. ToTensor: shape

# 4. Аугментации: покажите оригинал, flipped, rotated
# Подсказка: transforms работают с PIL-объектами (до ToTensor)

### Dataset и DataLoader

Стандартный способ организации данных в PyTorch:

```python
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, self.labels[idx]

loader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=2)
```

- `Dataset` — отвечает за один пример (загрузка + трансформация)
- `DataLoader` — формирует батчи, перемешивает, параллельная загрузка

---
## Полезные ссылки

- [Документация PyTorch](https://pytorch.org/docs/stable/)
- [PyTorch Tutorials](https://pytorch.org/tutorials/)
- [Таблица соответствия NumPy и PyTorch](https://github.com/torch/torch7/wiki/Torch-for-Numpy-users)
- [PyTorch на GPU](https://pytorch.org/docs/stable/notes/cuda.html)
- [PyTorch примеры](https://github.com/pytorch/examples)